# control_hub_v1

Project notebook. Each cell exercises one `hw_toolkit` module. Doctrine: **load-first** — actuators / sensors / MCU lock before power rails.

| cell | module | does |
|---|---|---|
| 1 | `hw_toolkit.board` | load + validate ResearchBundle |
| 2 | `hw_agent.core.research_bundle` | inspect subsystems + interfaces |
| 3 | `hw_toolkit.calc` | operating-point math |
| 4 | `hw_toolkit.schematic` _(pending)_ | active schematic object |
| 5 | `hw_toolkit.placement` _(via Board)_ | move plan |
| 6 | `hw_toolkit.bom` _(pending)_ | BOM dataframe + export |
| 7 | `hw_toolkit.routing` _(pending)_ | route + fab export |

## Cell 1 — load (`hw_toolkit.board`)

Validates `research_bundle.json` against the pydantic schema. Raises `BundleValidationError` on any field error.

In [ ]:
import hw_toolkit as hw
board = hw.Board.load("control_hub_v1")
board

## Cell 2 — inspect bundle (`hw_agent.core.research_bundle`)

Read-only view of what the researcher locked in.

In [ ]:
import pandas as pd
subs = pd.DataFrame([s.model_dump() for s in board.bundle.subsystems])
subs[["id", "category", "mpn", "package", "price_usd"]]

In [ ]:
ifaces = pd.DataFrame([i.model_dump() for i in board.bundle.interfaces])
ifaces[["id", "type", "from_subsystem", "from_port", "to_subsystem", "to_port", "voltage_nominal_v", "protocol"]]

## Cell 3 — buck math (`hw_toolkit.calc`)

Operating-point object. Stateless calcs return typed dataclasses.

In [ ]:
# rail_3v3 interface: vbat 11.1V -> mcu 3.3V @ ~0.5A budget
b = hw.calc.Buck(vin=11.1, vout=3.3, iout=0.5)
b

In [ ]:
b.inductor()

In [ ]:
b.output_cap(target_ripple_mv=30)

In [ ]:
# TPS54331DR rdson ~ 80 mOhm, theta_ja ~ 40 C/W (SOIC-8)
T = b.thermal(rdson_mohm=80, theta_ja=40)
T

In [ ]:
# fail-fast: __bool__ is tj_safe
assert T, f"thermal not safe: Tj={T.tj_c} C, margin={T.margin_c} C"

## Cell 4 — schematic (`hw_toolkit.schematic`) _(pending)_

Today: `board.write_schematic()` returns a plan; dispatch is manual via `designer-mcp` add_* tools (one tool call per op).

Target: `hw.Schematic` active object with `.add() / .connect() / .move() / .save()` and `_repr_png_` inline render.

In [ ]:
plan = board.write_schematic()
from collections import Counter
Counter(type(op).__name__ for op in plan.ops)

In [ ]:
# inspect first 3 ops as dict (what would be dispatched to designer-mcp)
plan.as_tool_calls()[:3]

## Cell 5 — placement (`hw_toolkit.board.place`)

Zone-based moves dispatched via `live-edit-mcp` `live_move_symbol`. Last op carries `with_render=True` so eeschema renders the final layout.

In [ ]:
place = board.place()
pd.DataFrame([
    {"ref": op.ref, "x_mm": op.x_mm, "y_mm": op.y_mm, "render": op.with_render}
    for op in place.ops
])

In [ ]:
board.zones()

## Cell 6 — BOM (`hw_toolkit.bom`) _(pending)_

Target:

```python
bom = board.schematic.bom()
bom               # _repr_html_ dataframe
bom.export("bom.csv")
bom.export("jlc")
```

Today: hand-roll via the bundle.

In [ ]:
bom = pd.DataFrame([
    {
        "refdes": f"U{i+1}",
        "mpn": s.mpn,
        "package": s.package,
        "qty": s.qty_per_board,
        "price_usd": s.price_usd,
        "lcsc": s.lcsc or "",
    }
    for i, s in enumerate(board.bundle.subsystems)
])
bom

## Cell 7 — routing + fab (`hw_toolkit.routing`) _(pending)_

Target:

```python
r = board.schematic.route()
r.run(engine="freerouting")     # auto-fallback to orthoroute
r.export_fab(rev="A")           # gerbers + drill under fab/rev_A/
```

Today: not wired. Use `router-mcp` direct: `mcp__router-mcp__route_board(...)`.